In [ ]:
!pip install -q \
    gradio \
    groq \
    sentence-transformers \
    faiss-cpu \
    pypdf \
    python-docx \
    pandas \
    openpyxl

In [ ]:
import os
import faiss
import gradio as gr
import numpy as np
import pandas as pd

from groq import Groq
from pypdf import PdfReader
from docx import Document

from sentence_transformers import SentenceTransformer

In [ ]:
GROQ_API_KEY = "SUA CHAVE AQUI"

In [ ]:
client = Groq(
    api_key=GROQ_API_KEY
)

embedding_model = SentenceTransformer(
    'sentence-transformers/all-MiniLM-L6-v2'
)

In [ ]:
def read_pdf(file_path):

    text = ""

    reader = PdfReader(file_path)

    for page in reader.pages:

        content = page.extract_text()

        if content:
            text += content + "\n"

    return text


def read_docx(file_path):

    doc = Document(file_path)

    return "\n".join(
        [p.text for p in doc.paragraphs]
    )


def read_xlsx(file_path):

    xls = pd.ExcelFile(file_path)

    text = ""

    for sheet in xls.sheet_names:

        df = pd.read_excel(
            file_path,
            sheet_name=sheet
        )

        text += f"\nPlanilha: {sheet}\n"

        text += df.astype(str).to_string()

    return text


def read_txt(file_path):

    with open(file_path, 'r', encoding='utf-8') as f:

        return f.read()

In [ ]:
def extract_text(file_path):

    extension = file_path.split('.')[-1].lower()

    if extension == 'pdf':
        return read_pdf(file_path)

    elif extension == 'docx':
        return read_docx(file_path)

    elif extension == 'xlsx':
        return read_xlsx(file_path)

    elif extension == 'txt':
        return read_txt(file_path)

    return ""

In [ ]:
def chunk_text(text, chunk_size=500):

    words = text.split()

    chunks = []

    for i in range(0, len(words), chunk_size):

        chunk = " ".join(words[i:i + chunk_size])

        chunks.append(chunk)

    return chunks

In [ ]:
all_chunks = []
metadata = []
index = None

In [ ]:
def process_documents(files):

    global all_chunks
    global metadata
    global index

    all_chunks = []
    metadata = []

    for file in files:

        text = extract_text(file.name)

        chunks = chunk_text(text)

        for chunk in chunks:

            all_chunks.append(chunk)

            metadata.append({
                "source": os.path.basename(file.name)
            })

    embeddings = embedding_model.encode(all_chunks)

    embeddings = np.array(embeddings).astype('float32')

    dimension = embeddings.shape[1]

    index = faiss.IndexFlatL2(dimension)

    index.add(embeddings)

    return f"{len(files)} documentos processados com sucesso."

In [ ]:
def search(query, top_k=3):

    query_embedding = embedding_model.encode([query])

    query_embedding = np.array(query_embedding).astype('float32')

    distances, indices = index.search(
        query_embedding,
        top_k
    )

    results = []

    for idx in indices[0]:

        results.append({
            "chunk": all_chunks[idx],
            "source": metadata[idx]["source"]
        })

    return results

In [ ]:
def ask_rag(question):

    results = search(question)

    context = ""

    sources = []

    for result in results:

        context += result["chunk"] + "\n"

        sources.append(result["source"])

    prompt = f"""
    Responda usando exclusivamente o contexto abaixo.

    CONTEXTO:
    {context}

    PERGUNTA:
    {question}
    """

    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ],
        temperature=0.2,
        max_tokens=1024
    )

    answer = response.choices[0].message.content

    fontes = "\n".join(set(sources))

    return f"""
    Resposta:
    {answer}

    Fontes:
    {fontes}
    """

In [ ]:
with gr.Blocks() as demo:

    gr.Markdown("# RAG com Groq + Llama 3.3")

    arquivos = gr.File(
        file_count="multiple",
        label="Envie os documentos"
    )

    processar_btn = gr.Button(
        "Processar Documentos"
    )

    status = gr.Textbox(
        label="Status"
    )

    pergunta = gr.Textbox(
        label="Pergunta"
    )

    perguntar_btn = gr.Button(
        "Perguntar"
    )

    resposta = gr.Textbox(
        label="Resposta",
        lines=15
    )

    processar_btn.click(
        fn=process_documents,
        inputs=arquivos,
        outputs=status
    )

    perguntar_btn.click(
        fn=ask_rag,
        inputs=pergunta,
        outputs=resposta
    )

demo.launch(debug=True)